# Swipe Atlas auto-sklearn Colab Run

Run this notebook in Google Colab to produce the strict auto-sklearn comparison. The main project stays Windows-friendly; this notebook creates a Linux/Python 3.9 conda environment just for auto-sklearn.

## 1. Clone The Project

This pulls the latest `aaron-staging` branch into the Colab runtime.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d WIA1006_MLPROJECT ]; then
  git clone -b aaron-staging https://github.com/aarntn/WIA1006_MLPROJECT.git
fi
cd WIA1006_MLPROJECT
git checkout aaron-staging
git pull


In [ ]:
%cd /content/WIA1006_MLPROJECT


## 2. Create The auto-sklearn Environment

This installs Miniforge and creates a Python 3.9 environment. It may take several minutes.

In [ ]:
%%bash
set -e

if [ ! -x /content/miniforge/bin/conda ]; then
  wget -q -O /content/miniforge.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
  bash /content/miniforge.sh -b -p /content/miniforge
fi

/content/miniforge/bin/conda config --set channel_priority strict

if ! /content/miniforge/bin/conda env list | grep -q "^autosklearn39 "; then
  /content/miniforge/bin/conda create -y -n autosklearn39 -c conda-forge \
    python=3.9 auto-sklearn=0.15.0 pandas numpy matplotlib seaborn joblib
fi


## 3. Verify The Install

In [ ]:
!/content/miniforge/bin/conda run -n autosklearn39 python -c "import sys, sklearn, autosklearn; print(sys.version); print('sklearn', sklearn.__version__); print('autosklearn', autosklearn.__version__)"


## 4. Run auto-sklearn

This uses the same 12,000-row sample as the existing AutoGluon comparison. The time limit is 3,600 seconds, so expect up to about one hour.

In [ ]:
!/content/miniforge/bin/conda run -n autosklearn39 python scripts/06_automl_comparison.py --backend autosklearn --sample 12000 --autosklearn-time 3600


## 5. Inspect Results

In [ ]:
import pandas as pd

results = pd.read_csv("reports/automl_results.csv")
display(results)

leaderboard = pd.read_csv("reports/automl_leaderboard_autosklearn.csv")
display(leaderboard.head(15))


## 6. Download Outputs

Download these files and place them back into the same paths in your local Windows repo.

In [ ]:
from google.colab import files

for path in [
    "reports/automl_results.csv",
    "reports/automl_leaderboard_autosklearn.csv",
    "reports/engagement_summary.md",
    "reports/figures/20_automl_comparison.png",
]:
    files.download(path)


## Report Sentence

Only use this after the auto-sklearn row has real numbers:

> auto-sklearn was executed in Google Colab/Linux because it cannot run on native Windows. Its holdout R2 was [fill], compared with AutoGluon's 0.000 and the tuned manual model's -0.003, showing that AutoML also could not extract meaningful predictive signal from the safe feature set.